# Applied Time Series Analysis: Final Project Walkthrough
## Multivariate Air Quality (AQI) Forecasting with VAR/VECM, SARIMAX, and Bootstrapped Uncertainty

This lab walkthrough executes each step of the econometric and machine learning modeling pipeline.

### Section 1: Environment & Configuration Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

from src.config import init_project, load_config
cfg = init_project(city="Delhi")
print(f"Project initialized for {cfg['default_city']}. Random seed: {cfg['project']['random_seed']}")

### Section 2: Data Acquisition & Quality Audit
Downloads Open-Meteo pollutant (CAMS) and archive weather data or generates synthetic data.

In [ ]:
from src.data_fetch import fetch_all_data
hourly_df, daily_df, is_synthetic = fetch_all_data(cfg, city="Delhi", allow_synthetic=True)
print(f"Daily Data Shape: {daily_df.shape}, Synthetic: {is_synthetic}")
daily_df.head(3)

### Section 3: Data Cleaning & Feature Engineering
Enforces regular index, caps outliers, interpolates short gaps, and builds log transforms & event flags.

In [ ]:
from src.data_clean import clean_data
from src.features import prepare_features

cleaned_df = clean_data(daily_df, cfg, city="Delhi", freq="D")
featured_df = prepare_features(cleaned_df, cfg)
print(f"Featured data contains {featured_df.shape[1]} columns.")

### Section 4: Exploratory Data Analysis & STL Decomposition

In [ ]:
from src.eda import run_eda
stl_metrics = run_eda(featured_df, cfg, is_synthetic=is_synthetic)
stl_metrics

### Section 5: Stationarity Analysis (ADF & KPSS Decisions)

In [ ]:
from src.stationarity import run_stationarity_tests
test_cols = ["pm2_5", "log_pm2_5", "temperature_2m", "wind_speed_10m"]
stat_table = run_stationarity_tests(featured_df, cols=test_cols)
stat_table[["variable", "transform", "adf_pvalue", "kpss_pvalue", "decision"]]

### Section 6: Univariate Baselines & SARIMAX with Weather Covariates

In [ ]:
from src.sarimax_model import run_sarimax
sarimax_results = run_sarimax(featured_df, cfg, target="pm2_5", horizon=7)
if sarimax_results:
    print(f"SARIMAX Optimal Order: {sarimax_results['order']} with AIC: {sarimax_results['aic']:.1f}")

### Section 7: Multivariate VAR / VECM Modeling

In [ ]:
from src.var_model import run_var_pipeline
var_res = run_var_pipeline(featured_df, cfg, horizon=7)
print(f"Model Selected: {var_res['model_choice']}, Stability: {var_res['is_stable']}, Cointegration Rank: {var_res['coint_rank']}")
var_res['forecast'].head()

### Section 8: Structural Analysis (Granger Causality, IRF, FEVD)

In [ ]:
from src.var_analysis import run_var_analysis
diff_endo = featured_df[var_res['endo_vars']].diff().dropna()
analysis_res = run_var_analysis(var_res['var_model'], diff_endo, cfg, is_synthetic=is_synthetic)
analysis_res['granger_matrix']

### Section 9: Bootstrapped Uncertainty Quantification

In [ ]:
from src.uncertainty import bagged_ets_forecast, residual_bootstrap_var
boot_var = residual_bootstrap_var(var_res['var_model'], steps=7, n_bootstrap=100)
print(f"Bootstrapped 95% Interval for PM2.5 (Horizon 1): [{boot_var['lower_95'][0, 0]:.2f}, {boot_var['upper_95'][0, 0]:.2f}]")

### Section 10: Rolling-Origin Cross-Validation & CPCB AQI Classification

In [ ]:
from src.evaluation import rolling_origin_cv
eval_res = rolling_origin_cv(featured_df, cfg, target_col="pm2_5", horizons=(1, 3, 7))
eval_res['model_comparison']